In [1]:
import warnings
warnings.filterwarnings("ignore")

import math
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB, BernoulliNB, GaussianNB
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
from sklearn.datasets import load_iris

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12


In [2]:
messages = [
    "team meeting today", "project meeting schedule", "can we discuss project",
    "please review the report", "lunch meeting with team", "project deadline today",
    "call me after meeting", "review project update", "send me the project file",
    "client meeting moved tomorrow", "the report needs review", "can you join the call",
    "free prize claim now", "win free cash now", "claim your lottery prize",
    "free offer only today", "urgent cash prize claim", "win money now",
    "exclusive free lottery offer", "claim cash reward today", "limited offer claim cash",
    "winner claim free reward", "urgent lottery winner", "free money offer now",
    "meeting notes attached", "deadline moved to tomorrow", "please send the update",
    "team lunch tomorrow", "project review call", "client report attached",
    "claim exclusive bonus", "cash bonus for winner", "free lottery cash",
    "urgent prize winner", "reward claim offer", "money cash prize",
]

labels = [
    "ham", "ham", "ham", "ham", "ham", "ham", "ham", "ham", "ham", "ham", "ham", "ham",
    "spam", "spam", "spam", "spam", "spam", "spam", "spam", "spam", "spam", "spam", "spam", "spam",
    "ham", "ham", "ham", "ham", "ham", "ham",
    "spam", "spam", "spam", "spam", "spam", "spam"
]

df = pd.DataFrame({"message": messages, "label": labels})
df


,message,label
0,team meeting today,ham
1,project meeting schedule,ham
2,can we discuss project,ham
3,please review the report,ham
4,lunch meeting with team,ham
5,project deadline today,ham
6,call me after meeting,ham
7,review project update,ham
8,send me the project file,ham
9,client meeting moved tomorrow,ham


In [7]:
def tokenize(text):
    return text.lower().strip().split()

vocab = []
for message in messages:
    for word in tokenize(message):
        vocab.append(word)
vocab = sorted(set(vocab))

In [8]:
len(vocab)

48

In [13]:
def make_bow_matrix(texts, vocab):
    rows = []
    for text in texts:
        counts = Counter(tokenize(text))
        rows.append([counts[word] for word in vocab])
    return pd.DataFrame(rows, columns=vocab)

bow_matrix = make_bow_matrix(messages, vocab)
bow_matrix.insert(0, "label", labels)
bow_matrix.insert(0, "message", messages)
bow_matrix

,message,label,after,attached,bonus,call,can,cash,claim,client,...,today,tomorrow,update,urgent,we,win,winner,with,you,your
0,team meeting today,ham,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
1,project meeting schedule,ham,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,can we discuss project,ham,0,0,0,0,1,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,please review the report,ham,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,lunch meeting with team,ham,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
5,project deadline today,ham,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
6,call me after meeting,ham,1,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,review project update,ham,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
8,send me the project file,ham,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,client meeting moved tomorrow,ham,0,0,0,0,0,0,0,1,...,0,1,0,0,0,0,0,0,0,0


In [15]:
class_counts = df["label"].value_counts()
priors = class_counts / len(df)
priors

,count
label,
ham,0.5
spam,0.5


In [17]:
word_counts_by_class = {}
for c in sorted(df["label"].unique()):
    all_words = []
    for msg in df.loc[df["label"] == c, "message"]:
        all_words.extend(tokenize(msg))
    word_counts_by_class[c] = Counter(all_words)
word_counts_by_class

{'ham': Counter({'team': 3,
          'meeting': 6,
          'today': 2,
          'project': 6,
          'schedule': 1,
          'can': 2,
          'we': 1,
          'discuss': 1,
          'please': 2,
          'review': 4,
          'the': 5,
          'report': 3,
          'lunch': 2,
          'with': 1,
          'deadline': 2,
          'call': 3,
          'me': 2,
          'after': 1,
          'update': 2,
          'send': 2,
          'file': 1,
          'client': 2,
          'moved': 2,
          'tomorrow': 3,
          'needs': 1,
          'you': 1,
          'join': 1,
          'notes': 1,
          'attached': 2,
          'to': 1}),
 'spam': Counter({'free': 7,
          'prize': 5,
          'claim': 8,
          'now': 4,
          'win': 2,
          'cash': 7,
          'your': 1,
          'lottery': 4,
          'offer': 5,
          'only': 1,
          'today': 2,
          'urgent': 3,
          'money': 3,
          'exclusive': 2,
          'rew

In [18]:
count_table = pd.DataFrame(word_counts_by_class).fillna(0).astype(int)
count_table["total"] = count_table.sum(axis=1)
count_table.sort_values("total", ascending=False).head(20)

,ham,spam,total
claim,0,8,8
free,0,7,7
cash,0,7,7
meeting,6,0,6
project,6,0,6
prize,0,5,5
the,5,0,5
offer,0,5,5
lottery,0,4,4
winner,0,4,4


In [26]:
total_words_by_class = {c: sum(word_counts_by_class[c].values()) for c in word_counts_by_class}

alpha = 1.0
def likelihood_with_smoothing(word, c, alpha=1.0):
    numerator = word_counts_by_class[c][word] + alpha
    denominator = total_words_by_class[c] + alpha * len(vocab)
    return numerator / denominator

likelihood_rows = []
for word in vocab:
    likelihood_rows.append({
        "word": word,
        "P(word|spam)": likelihood_with_smoothing(word, "spam"),
        "P(word|ham)": likelihood_with_smoothing(word, "ham")
    })

likelihood_df = pd.DataFrame(likelihood_rows)
likelihood_df.head(20)

,word,P(word|spam),P(word|ham)
0,after,0.008850,0.017544
1,attached,0.008850,0.026316
2,bonus,0.026549,0.008772
3,call,0.008850,0.035088
4,can,0.008850,0.026316
5,cash,0.070796,0.008772
6,claim,0.079646,0.008772
7,client,0.008850,0.026316
8,deadline,0.008850,0.026316
9,discuss,0.008850,0.017544


In [38]:
word_a, word_b, c = "free", "prize", "spam"
mask = (likelihood_df["word"] == word_a) | (likelihood_df["word"] == word_b)
result_ham = 1.0
for i in likelihood_df[mask]['P(word|ham)'] * priors["ham"]:
    result_ham *= i


result_spam = 1.0
for i in likelihood_df[mask]['P(word|spam)'] * priors["spam"]:
    result_spam *= i
print(result_ham)
print(result_spam)
print(max(result_ham, result_spam))

1.9236688211757463e-05
0.0009397760200485551
0.0009397760200485551


In [39]:
word_a, word_b, c = "free", "prize", "spam"
mask = (likelihood_df["word"] == word_a) | (likelihood_df["word"] == word_b)
result_ham = 0.0
for i in likelihood_df[mask]['P(word|ham)'] * priors["ham"]:
    result_ham += math.log(i)

result_spam = 0.0
for i in likelihood_df[mask]['P(word|spam)'] * priors["spam"]:
    result_spam += math.log(i)
print(result_ham)
print(result_spam)
print(max(result_ham, result_spam))

-10.858691257908882
-6.969868987636681
-6.969868987636681


In [42]:
X_train, X_test, y_train, y_test = train_test_split(
    df["message"], df["label"],
    test_size=0.30,
    random_state=42,
    stratify=df["label"]
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Train class counts:")
print(pd.Series(y_train).value_counts())
print("Test class counts:")
print(pd.Series(y_test).value_counts())


Train size: 25
Test size: 11
Train class counts:
label
spam    13
ham     12
Name: count, dtype: int64
Test class counts:
label
ham     6
spam    5
Name: count, dtype: int64


In [43]:
vectorizer = CountVectorizer()
X_train_counts = vectorizer.fit_transform(X_train)
X_test_counts = vectorizer.transform(X_test)
print(X_train_counts.shape)
print(X_test_counts.shape)

(25, 41)
(11, 41)


In [45]:
sk_model = MultinomialNB()
sk_model.fit(X_train_counts, y_train)

y_pred = sk_model.predict(X_test_counts)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         ham       1.00      1.00      1.00         6
        spam       1.00      1.00      1.00         5

    accuracy                           1.00        11
   macro avg       1.00      1.00      1.00        11
weighted avg       1.00      1.00      1.00        11



In [47]:
feature_names = np.array(vectorizer.get_feature_names_out())
feature_names

array(['attached', 'bonus', 'call', 'can', 'cash', 'claim', 'client',
       'deadline', 'discuss', 'exclusive', 'file', 'for', 'free', 'join',
       'limited', 'lottery', 'lunch', 'me', 'meeting', 'money', 'moved',
       'offer', 'only', 'please', 'prize', 'project', 'report', 'review',
       'reward', 'schedule', 'send', 'team', 'the', 'to', 'today',
       'tomorrow', 'urgent', 'we', 'winner', 'you', 'your'], dtype=object)

In [48]:
vocab

['after',
 'attached',
 'bonus',
 'call',
 'can',
 'cash',
 'claim',
 'client',
 'deadline',
 'discuss',
 'exclusive',
 'file',
 'for',
 'free',
 'join',
 'limited',
 'lottery',
 'lunch',
 'me',
 'meeting',
 'money',
 'moved',
 'needs',
 'notes',
 'now',
 'offer',
 'only',
 'please',
 'prize',
 'project',
 'report',
 'review',
 'reward',
 'schedule',
 'send',
 'team',
 'the',
 'to',
 'today',
 'tomorrow',
 'update',
 'urgent',
 'we',
 'win',
 'winner',
 'with',
 'you',
 'your']

In [50]:
# Optional real text dataset. This requires internet the first time it runs.
try:
    from sklearn.datasets import fetch_20newsgroups
    categories = ["sci.space", "rec.sport.baseball"]
    train_news = fetch_20newsgroups(subset="train", categories=categories, remove=("headers", "footers", "quotes"))
    test_news = fetch_20newsgroups(subset="test", categories=categories, remove=("headers", "footers", "quotes"))

    print("Train documents:", len(train_news.data))
    print("Test documents:", len(test_news.data))
    print("Classes:", train_news.target_names)
    print("Sample document:")
    print(train_news.data[0][:600])
    print("Class:", train_news.target_names[train_news.target[0]])
except Exception as e:
    print("Could not load 20 Newsgroups dataset.")
    print("Reason:", e)


Train documents: 1190
Test documents: 791
Classes: ['rec.sport.baseball', 'sci.space']
Sample document:
I've been saying this for quite some time, but being absent from the
net for a while I figured I'd stick my neck out a bit...

The Royals will set the record for fewest runs scored by an AL
team since the inception of the DH rule.  (p.s. any ideas what this is?)

They will fall easily short of 600 runs, that's for damn sure.  I can't
believe these media fools picking them to win the division (like our
Tom Gage of the Detroit News claiming Herk Robinson is some kind of
genius for the trades/aquisitions he's made)

c-ya

Sean


Class: rec.sport.baseball


In [52]:
news_vectorizer = CountVectorizer(stop_words="english")
X_train_news_counts = news_vectorizer.fit_transform(train_news.data)
X_test_news_counts = news_vectorizer.transform(test_news.data)

news_nb = MultinomialNB()
news_nb.fit(X_train_news_counts, train_news.target)

y_pred = news_nb.predict(X_test_news_counts)
print(classification_report(test_news.target, y_pred))


              precision    recall  f1-score   support

           0       0.92      0.96      0.94       397
           1       0.95      0.91      0.93       394

    accuracy                           0.94       791
   macro avg       0.94      0.94      0.94       791
weighted avg       0.94      0.94      0.94       791

